In [7]:
import pandas as pd
import numpy as np

In [8]:
# =====================================================
# Load Dataset & Basic Preprocessing
# =====================================================

df = pd.read_csv('../Data/data.csv')
df = df[df['Open'] == 1]

df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)   

df['StateHoliday'] = df['StateHoliday'].astype(str)

df.drop("DayOfWeek", axis=1, inplace=True)
df.drop("Open", axis=1, inplace=True)
df.drop("Customers", axis=1, inplace=True)

D:\Temp\ipykernel_18612\1081138726.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../Data/data.csv')


In [9]:
df['StateHoliday'] = df['StateHoliday'].astype(str).map({
    '0' : 0,
    'a' : 1,
    'b' : 2,
    'c' : 3
})

print(df['StateHoliday'].value_counts())
print(df['StateHoliday'].dtype)

StateHoliday
0    647692
1       501
2        96
3        71
Name: count, dtype: int64
int64


In [10]:
df['lag_1'] = df['Sales'].shift(1)
df['lag_7'] = df['Sales'].shift(7)
df['lag_30'] = df['Sales'].shift(30)

In [11]:
df['rolling_7_mean'] = df['Sales'].shift(1).rolling(7).mean()
df['rolling_7_std'] = df['Sales'].shift(1).rolling(7).std()

df['rolling_14_mean'] = df['Sales'].shift(1).rolling(14).mean()
df['rolling_14_std'] = df['Sales'].shift(1).rolling(14).std()

df['rolling_30_mean'] = df['Sales'].shift(1).rolling(30).mean()
df['rolling_30_std'] = df['Sales'].shift(1).rolling(30).std()

In [12]:
df['day_of_month'] = df.index.day

df["day_of_week"] = df.index.dayofweek
df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

df["month"] = df.index.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df["quarter"] = df.index.quarter
df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)

df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

In [13]:
df['expanding_sum'] = df['Sales'].expanding().sum()
df['expanding_mean'] = df['Sales'].expanding().mean()
df['expanding_std'] = df['Sales'].expanding().std()

In [14]:
print(f"Total features created: {len(df.columns)}") 
print("\nFeature columns:")
print(df.columns.tolist())

Total features created: 28

Feature columns:
['Store', 'Sales', 'Promo', 'StateHoliday', 'SchoolHoliday', 'lag_1', 'lag_7', 'lag_30', 'rolling_7_mean', 'rolling_7_std', 'rolling_14_mean', 'rolling_14_std', 'rolling_30_mean', 'rolling_30_std', 'day_of_month', 'day_of_week', 'day_sin', 'day_cos', 'month', 'month_sin', 'month_cos', 'quarter', 'quarter_sin', 'quarter_cos', 'is_weekend', 'expanding_sum', 'expanding_mean', 'expanding_std']


In [15]:
nan_count = df.isnull().sum()
print(nan_count[nan_count > 0])

lag_1               1
lag_7               7
lag_30             30
rolling_7_mean      7
rolling_7_std       7
rolling_14_mean    14
rolling_14_std     14
rolling_30_mean    30
rolling_30_std     30
expanding_std       1
dtype: int64


In [16]:
print(f"\nOriginal rows : {len(df)}")

df = df.dropna()
print(f"After dropna  : {len(df)}")
print(f"Rows removed  : {len(df) - len(df)}")
print(f"Data loss %   : {((len(df) - len(df)) / len(df)) * 100:.1f}%")


Original rows : 648360


After dropna  : 648330
Rows removed  : 0
Data loss %   : 0.0%


In [17]:
corr = df.corr(numeric_only=True)
corr['Sales'].sort_values(ascending=False)

Sales              1.000000
rolling_30_mean    0.479473
rolling_14_mean    0.461903
rolling_7_mean     0.419723
Promo              0.366653
lag_1              0.270025
lag_7              0.269146
rolling_30_std     0.253723
lag_30             0.220720
rolling_14_std     0.216056
rolling_7_std      0.182448
day_cos            0.126993
expanding_std      0.108614
month              0.093630
expanding_mean     0.091844
quarter            0.079781
day_sin            0.059926
quarter_cos        0.058083
month_cos          0.042496
SchoolHoliday      0.025940
StateHoliday       0.018172
Store              0.009262
month_sin          0.006945
quarter_sin       -0.010215
day_of_month      -0.052727
expanding_sum     -0.074069
is_weekend        -0.152306
day_of_week       -0.177675
Name: Sales, dtype: float64

In [18]:
features = [
    # Store
    'Store',
    'Promo',
    'StateHoliday',

    # Calendar
    'day_of_week',
    'month',
    'is_weekend',

    # Lag
    'lag_1',
    'lag_7',
    'lag_30',

    # Rolling
    'rolling_7_mean',
    'rolling_14_mean',
    'rolling_30_mean',
]

In [20]:
cols_to_save = features + ['Sales']

df_save = df[cols_to_save].copy()

# Save
df_save.to_csv('rossmann_features.csv', index=True)  

print(f"✅ Saved!")
print(f"Shape: {df_save.shape}")
print(df_save.head())

✅ Saved!
Shape: (648330, 13)
            Store  Promo  StateHoliday  day_of_week  month  is_weekend  \
Date                                                                     
2014-12-31     35      0             0            2     12           0   
2014-12-31     37      0             0            2     12           0   
2014-12-31     38      0             0            2     12           0   
2014-12-31     39      0             0            2     12           0   
2014-12-31     40      0             0            2     12           0   

             lag_1   lag_7   lag_30  rolling_7_mean  rolling_14_mean  \
Date                                                                   
2014-12-31  5096.0  5385.0   2605.0     4208.571429      4443.928571   
2014-12-31  5187.0  1713.0   2269.0     4180.285714      4508.285714   
2014-12-31  2920.0  4890.0   3804.0     4352.714286      4447.428571   
2014-12-31  3534.0  2257.0  10152.0     4159.000000      4482.500000   
2014-12-31  2916.0  